<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Ek F: LLM Değerlendirmesine Yaygın Yaklaşımlar

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.0
torch version: 2.7.1
tokenizers version: 0.21.2


&nbsp;
## F.1 LLM'ler için başlıca değerlendirme yöntemlerini anlamak

- Bu bölümde kod yok

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F01_raschka.webp" width="500px">

&nbsp;
### F.2 Şık doğruluğunu değerlendirmek

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F02_raschka.webp" width="500px">

- Bu şeklin, üretilen çıktı harfini doğru yanıt harfiyle karşılaştırdığımız çoktan seçmeli tabanlı bir değerlendirmenin (MMLU gibi) basitleştirilmiş bir sürümünü gösterdiğini unutmayın
- Pratikte bunun çeşitleri arasında log-olasılık puanlaması da vardır; burada yalnızca son harfi kontrol etmek yerine, modelin her aday yanıtı ne kadar olası gördüğünü hesaplarız
- Akıl yürütme modellerinde bu, modele verildiğinde doğru yanıtın üretilme olasılığını değerlendirmeyi de içerebilir
- Her iki durumda da değerlendirme, modelin önceden tanımlanmış yanıtlardan birini seçip seçmediğini kontrol eder
- (Çıktı olasılık puanları, metin üretme fonksiyonunu iyileştirdiğimiz 4. bölümde daha ayrıntılı ele alınıyor)

&nbsp;
#### F.2.1 Modeli yüklemek

In [2]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()
torch.set_float32_matmul_precision("high")

# Uyumluluk sorunları yaşıyorsanız aşağıdaki satırı
# yorumdan çıkarıp not defterini yeniden çalıştırmayı deneyin
# device = "cpu"

WHICH_MODEL = "base"

if WHICH_MODEL == "base":

    download_qwen3_small(
        kind="base", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-base.json"
    model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
    tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

elif WHICH_MODEL == "reasoning":

    download_qwen3_small(
        kind="reasoning", tokenizer_only=False, out_dir="qwen3"
    )

    tokenizer_path = Path("qwen3") / "tokenizer-reasoning.json"
    model_path = Path("qwen3") / "qwen3-0.6B-reasoning.pth"
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_path,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=True,
    )

else:
    raise ValueError(f"Invalid choice: WHICH_MODEL={WHICH_MODEL}")


model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device)


USE_COMPILE = False  # Set to true to enable compilation
if USE_COMPILE:
  torch._dynamo.config.allow_unspec_int_on_nn_module = True
  model = torch.compile(model)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
✓ qwen3/tokenizer-base.json already up-to-date


&nbsp;
#### F.2.2 Üretilen yanıt harfini kontrol etmek

In [3]:
example = {
    "question": (
        "How many ways are there to put 4 distinguishable"
        " balls into 2 indistinguishable boxes?"
    ),
    "choices": ["7", "11", "16", "8"],
    "answer": "D",
}

def format_prompt(example):
    return (
        f"{example['question']}\n"
        f"A. {example['choices'][0]}\n"
        f"B. {example['choices'][1]}\n"
        f"C. {example['choices'][2]}\n"
        f"D. {example['choices'][3]}\n"
        "Answer: "  # trailing space encourages a single-letter next token
    )

prompt = format_prompt(example)
print(prompt)

How many ways are there to put 4 distinguishable balls into 2 indistinguishable boxes?
A. 7
B. 11
C. 16
D. 8
Answer: 


---


- MMLU veri kümesindeki örnekleri doğrudan `datasets` kütüphanesiyle yükleyebilirsiniz (`pip install datasets` ya da `uv add datasets` ile kurulabilir):

```python
from datasets import load_dataset

configs = get_dataset_config_names("cais/mmlu")
dataset = load_dataset("cais/mmlu", "high_school_mathematics")

# Inspect the first example from test set:
example = dataset["test"][0]
print(example)
```

- Yukarıda `"high_school_mathematics"` alt kümesini kullandık; diğer alt kümelerin listesini almak için şu kodu kullanın:


```python
from datasets import get_dataset_config_names

subsets = get_dataset_config_names("cais/mmlu")
print(subsets)
```

---

In [4]:
prompt_ids = tokenizer.encode(prompt)
prompt_fmt = torch.tensor(prompt_ids, device=device).unsqueeze(0)

- Birkaç token üretiyor ve modelin yazdırdığı ilk A/B/C/D harfini ayıklıyoruz:

In [5]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def predict_choice(
    model, tokenizer, prompt_fmt, max_new_tokens=8
):
    pred = None
    for t in generate_text_basic_stream_cache(
        model=model,
        token_ids=prompt_fmt,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        answer = tokenizer.decode(t.squeeze(0).tolist())
        for letter in answer:
            letter = letter.upper()
            if letter in "ABCD":
                pred = letter
                break
        if pred:  # stop as soon as a letter appears
            break
    return pred

In [6]:
pred1 = predict_choice(model, tokenizer, prompt_fmt)

print(
    f"Generated letter: {pred1}\n"
    f"Correct? {pred1 == example['answer']}"
)

Generated letter: C
Correct? False


&nbsp;
### F.3 Yanıtları kontrol etmek için doğrulayıcılar kullanmak

- Bu bölümde kod yok (bkz. 3. bölüm)

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F03_raschka.webp" width="500px">

<br>
&nbsp;

### F.4 Modelleri tercihler ve liderlik tabloları ile karşılaştırmak

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F04_raschka.webp" width="500px">

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F05_raschka.webp" width="500px">

- Satranç sıralamalarından esinlenen Elo derecelendirmesi ("400 algoritması"): https://en.wikipedia.org/wiki/Performance_rating_(chess)
- LM Arena'nın, Elo benzeri bir ölçekte puan veren istatistiksel bir Bradely-Terry modeline geçtiğini unutmayın; ancak aynı ikili sıralama kavramı hâlâ geçerlidir

In [7]:
# İlk modelin kazanan, ikinci modelin kaybeden olduğu
# ikili "arena oyları"
votes = [
    ("GPT-5", "Claude-3"),  # First match-up: GPT-5 was preferred over Claude-3
    ("GPT-5", "Llama-4"),
    ("Claude-3", "Llama-3"),
    ("Llama-4", "Llama-3"),
    ("Claude-3", "Llama-3"),
    ("GPT-5", "Llama-3"),
]

In [8]:
def elo_ratings(vote_pairs, k_factor=32, initial_rating=1000):
    # Tüm modelleri aynı temel derecelendirmeyle başlat
    ratings = {
        model: initial_rating
        for pair in vote_pairs
        for model in pair
    }

    # Her maçtan sonra derecelendirmeleri güncelle
    for winner, loser in vote_pairs:

        # Derecelendirmeler verildiğinde mevcut kazanan için beklenen puan
        expected_winner = 1.0 / (
            1.0 + 10 ** ((ratings[loser] - ratings[winner]) / 400.0)
        )

        # k_factor, derecelendirme güncellemelerinin duyarlılığını belirler
        ratings[winner] = (
            ratings[winner] + k_factor * (1 - expected_winner)
        )
        ratings[loser] = (
            ratings[loser] + k_factor * (0 - (1 - expected_winner))
        )

    return ratings

In [9]:
ratings = elo_ratings(votes, k_factor=32, initial_rating=1000)

for model in sorted(ratings, key=ratings.get, reverse=True):
    print(f"{model:8s} : {ratings[model]:.1f}")

GPT-5    : 1043.7
Claude-3 : 1015.2
Llama-4  : 1000.7
Llama-3  : 940.4


- Beklenen kazanan puanı şöyle hesaplanır:

$$\text{expected\_winner} \;=\; \frac{1}{1 + 10^{\tfrac{\text{rating\_loser} - \text{rating\_winner}}{400}}}
$$

- Sezgi:
    - rating_winner >> rating_loser ise:
       - üs → çok negatif
       - payda ≈ 1
       - expected_winner ≈ 1 (neredeyse kesin galibiyet)
    - rating_winner << rating_loser ise:
       - üs → çok pozitif
       - payda → çok büyük
       - expected_winner ≈ 0 (neredeyse kesin mağlubiyet)
    - rating_winner == rating_loser ise:
       - üs = 0
       - payda = 2
       - expected_winner = 0.5 (denk maç)

&nbsp;
### F.5 Yanıtları başka LLM'lerle değerlendirmek

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-f/Appendix_F_F06_raschka.webp" width="500px">

- Bu bölümde, ince ayarlı LLM'in yanıt değerlendirmesini başka ve daha büyük bir LLM kullanarak otomatikleştiriyoruz
- Özellikle, ollama ([https://ollama.com](https://ollama.com)) aracılığıyla yerelde çalıştırılabilen, Open AI'ın talimat ince ayarlı 20 milyar parametreli gpt-oss modelini kullanıyoruz

- Ollama, LLM'leri verimli şekilde çalıştırmaya yarayan açık kaynaklı bir uygulamadır
- Verimliliği en üst düzeye çıkarmak için LLM'leri saf C/C++ ile uygulayan llama.cpp ([https://github.com/ggerganov/llama.cpp](https://github.com/ggerganov/llama.cpp)) etrafında bir sarmalayıcıdır
- Bunun, LLM'leri eğitmek veya ince ayar yapmak için değil, metin üretmek (çıkarım) için kullanılan bir araç olduğunu unutmayın
- Aşağıdaki kodu çalıştırmadan önce [https://ollama.com](https://ollama.com) adresini ziyaret edip talimatları izleyerek ollama'yı kurun (örneğin "Download" düğmesine tıklayıp işletim sisteminize uygun ollama uygulamasını indirin)

- macOS ve Windows kullanıcıları indirdikleri ollama uygulamasına tıklasın; komut satırı kullanımını kurmanızı isterse "evet" deyin
- Linux kullanıcıları ollama web sitesinde verilen kurulum komutunu kullanabilir
- Ollama'yı bilgisayarımızda çalıştırmanın 3 yolu var:

**1. `ollama serve`**

- Bu, ollama arka ucunu genellikle `http://localhost:11434` üzerinde bir sunucu olarak çalıştırır. API üzerinden çağırana kadar bir model yüklemez. Ollama'yı Python üzerinden kullanmak istiyorsak istediğimiz budur.

**2. `ollama run gpt-oss:20b`**

- Bu bir kolaylık sarmalayıcısıdır. Sunucu çalışmıyorsa onu başlatır, ardından modeli indirir (ilk seferde) ve bizi modelle sohbet edebileceğimiz etkileşimli bir terminale bırakır. Arka planda aynı sunucu API'sini kullanır.

**3. Ollama masaüstü uygulaması**

- Bu, aynı arka ucu otomatik olarak çalıştırır ve üzerine bir grafik arayüz sunar (yukarıdaki şekilde gösterildiği gibi).
Ayrıca varsayılanlar (sistem istemi, sıcaklık, durdurma dizileri) uygular; bu da yanıtların ham API kullanımından neden farklı göründüğünü açıklayabilir.

---

**Not**:

- Yukarıda anlatıldığı gibi terminalde `ollama serve` çalıştırırken `Error: listen tcp 127.0.0.1:11434: bind: address already in use` şeklinde bir hata mesajıyla karşılaşabilirsiniz
- Böyle bir durumda `OLLAMA_HOST=127.0.0.1:11435 ollama serve` komutunu deneyin (bu adres de kullanımdaysa, kullanımda olmayan bir adres bulana kadar sayıyı birer artırmayı deneyin)

---

- Örneğin ollama'yı denemek için, 20 milyar parametreli gpt-oss 20B modelini denemek üzere `ollama run gpt-oss:20b` kullanabiliriz. Model
  (yaklaşık 13 GB) bu komutu ilk çalıştırdığınızda otomatik olarak
  indirilecektir. (Alternatif olarak, önceki şekle benzer şekilde masaüstü uygulamasında da kullanabilirsiniz.)
  
```bash
ollama run gpt-oss:20b
```


- Çıktı şöyle görünür:

```
$ ollama run gpt-oss:20b
pulling manifest 
pulling b112e727c6f1: 100% ▕█████████████████████████████████▏  13 GB                         
pulling fa6710a93d78: 100% ▕█████████████████████████████████▏ 7.2 KB                         
pulling f60356777647: 100% ▕█████████████████████████████████▏  11 KB                         
pulling d8ba2f9a17b3: 100% ▕█████████████████████████████████▏   18 B                         
pulling 55c108d8e936: 100% ▕█████████████████████████████████▏  489 B                         
verifying sha256 digest 
writing manifest 
removing unused layers 
success
```

- gpt-oss hakkında daha fazla bilgi için ayrıntılı yazıma bakın: [From GPT-2 to gpt-oss: Analyzing the Architectural Advances](https://magazine.sebastianraschka.com/p/from-gpt-2-to-gpt-oss-analyzing-the) 
- Ollama'yı `"gpt-oss:20b"` modeliyle (20B parametreli bir model) kullanmak 13 GB RAM gerektirir; makineniz bunu desteklemiyorsa, yalnızca yaklaşık 4 GB RAM gerektiren 4B parametreli `qwen3:4b` gibi daha küçük bir modeli deneyebilirsiniz
- Alternatif olarak, makineniz destekliyorsa daha büyük 120 milyarlık gpt-oss (`qwen3:235b`) ya da 235 milyar parametreli Qwen3 modelini (`qwen3:235b`) de kullanabilirsiniz
- İndirme tamamlandıktan sonra, modelle sohbet etmenizi sağlayan bir komut satırı istemi göreceksiniz
- "What is 1+2?" gibi bir istem deneyin; şuna benzer bir çıktı vermelidir

```
>>> What is 1+2?
Thinking...
User asks: "What is 1+2?" This is simple: answer 3. Provide explanation? Possibly ask for simple 
arithmetic. Provide answer: 3.
...done thinking.

1 + 2 = **3**
```

- Bu oturumu `/bye` girdisiyle sonlandırabilirsiniz

- Aşağıdaki kod, önceki bölümde ürettiğimiz test kümesi yanıtlarını değerlendirmek için ollama'yı kullanmaya geçmeden önce ollama oturumunun düzgün çalışıp çalışmadığını kontrol eder

In [10]:
import psutil

def check_if_running(process_name):
    running = False
    for proc in psutil.process_iter(["name"]):
        if process_name in proc.info["name"]:
            running = True
            break
    return running

ollama_running = check_if_running("ollama")

if not ollama_running:
    raise RuntimeError(
        "Ollama not running. Launch ollama before proceeding."
    )
print("Ollama running:", check_if_running("ollama"))

Ollama running: True


- Şimdi, modelle etkileşim kurmak için daha önce kullandığımız `ollama run` komutuna alternatif bir yol, aşağıdaki fonksiyon aracılığıyla Python'da REST API'sini kullanmaktır
- Bu not defterindeki sonraki hücreleri çalıştırmadan önce ollama'nın hâlâ çalıştığından emin olun (önceki kod hücreleri `"Ollama running: True"` yazdırmalıydı)
- Ardından modeli sorgulamak için aşağıdaki kod hücresini çalıştırın

In [11]:
import json
import requests


def query_model(
    prompt,
    model="gpt-oss:20b",
    # OLLAMA_HOST=127.0.0.1:11435 ollama serve kullandıysanız
    # adresi 11434'ten 11435'e güncelleyin
    url="http://localhost:11434/api/chat"
):
    # Veri yükünü bir sözlük olarak oluştur
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {     # Settings below are required for deterministic responses
            "seed": 123,
            "temperature": 0,
            "num_ctx": 2048
        }
    }

    # POST isteğini gönder
    with requests.post(url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines(decode_unicode=True):
            if not line:
                continue
            response_json = json.loads(line)
            if "message" in response_json:
                response_data += response_json["message"]["content"]

    return response_data

In [12]:
ollama_model = "gpt-oss:20b"
result = query_model("What is 1+2?", ollama_model)
print(result)

3


- Şimdi, yukarıda tanımladığımız `query_model` fonksiyonunu kullanarak kendi modelimizin yanıtlarını değerlendirebiliriz

In [16]:
def rubric_prompt(instruction, reference_answer, model_answer):
    rubric = (
        "You are a fair judge assistant. You will be given an instruction, "
        "a reference answer, and a candidate answer to evaluate, according "
        "to the following rubric:\n\n"
        "1: The response fails to address the instruction, providing "
        "irrelevant, incorrect, or excessively verbose content.\n"
        "2: The response partially addresses the instruction but contains "
        "major errors, omissions, or irrelevant details.\n"
        "3: The response addresses the instruction to some degree but is "
        "incomplete, partially correct, or unclear in places.\n"
        "4: The response mostly adheres to the instruction, with only "
        "minor errors, omissions, or lack of clarity.\n"
        "5: The response fully adheres to the instruction, providing a "
        "clear, accurate, and relevant answer in a concise and efficient "
        "manner.\n\n"
        "Now here is the instruction, the reference answer, and the "
        "response.\n"
    )

    prompt = (
        f"{rubric}\n"
        f"Instruction:\n{instruction}\n\n"
        f"Reference Answer:\n{reference_answer}\n\n"
        f"Answer:\n{model_answer}\n\n"
        f"Evaluation: "
    )
    return prompt

- `model_answer`, kendi modelimizin ürettiği yanıt olabilir; burada basitlik adına olası bir model yanıtını sabit olarak yazıyoruz

In [17]:
rendered_prompt = rubric_prompt(
    instruction=(
        "If all birds can fly, and a penguin is a bird, "
        "can a penguin fly?"
    ),
    reference_answer=(
        "Yes, according to the premise that all birds can fly, "
        "a penguin can fly."
    ),
    model_answer=(
        "Yes – under those premises a penguin would be able to fly."
    )
)
print(rendered_prompt)

You are a fair judge assistant. You will be given an instruction, a reference answer, and a candidate answer to evaluate, according to the following rubric:

1: The response fails to address the instruction, providing irrelevant, incorrect, or excessively verbose content.
2: The response partially addresses the instruction but contains major errors, omissions, or irrelevant details.
3: The response addresses the instruction to some degree but is incomplete, partially correct, or unclear in places.
4: The response mostly adheres to the instruction, with only minor errors, omissions, or lack of clarity.
5: The response fully adheres to the instruction, providing a clear, accurate, and relevant answer in a concise and efficient manner.

Now here is the instruction, the reference answer, and the response.

Instruction:
If all birds can fly, and a penguin is a bird, can a penguin fly?

Reference Answer:
Yes, according to the premise that all birds can fly, a penguin can fly.

Answer:
Yes – 

In [18]:
result = query_model(rendered_prompt, ollama_model)
print(result)

**Score: 5**

The candidate answer directly addresses the question, correctly applies the given premises, and concisely states that a penguin would be able to fly. It is accurate, relevant, and clear.
